# Patrón Estructural: Facade

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Facade** ofrece una **interfaz simple y unificada** a un subsistema complejo,
ocultando al cliente los detalles de sus múltiples componentes.

### ¿Qué problema resuelve en la banca?
Hacer una **transferencia** no es un solo paso: hay que validar el saldo, pasar el control
**antifraude**, **debitar/acreditar** las cuentas, **registrar** el movimiento en el libro
contable y **notificar** al cliente. Si cada parte del sistema (app, cajero, web) tiene que
orquestar todos esos subsistemas a mano, el código se duplica y es fácil olvidar un paso.
El Facade expone un único método `transferir(...)`.

## Código *sin patrón* (el problema es evidente)
El cliente debe conocer y coordinar **todos** los subsistemas en el orden correcto.

In [1]:
class ValidadorSaldo:
    def hay_fondos(self, origen, monto): return origen["saldo"] >= monto

class Antifraude:
    def es_seguro(self, origen, destino, monto): return monto <= 10_000_000

class Libro:
    def registrar(self, origen, destino, monto):
        print(f"   [contabilidad] registrado: {origen['id']}->{destino['id']} ${monto}")

class Notificador:
    def enviar(self, cuenta, texto):
        print(f"   [notif] a {cuenta['id']}: {texto}")


# El CLIENTE tiene que orquestar todo a mano, en el orden correcto
val = ValidadorSaldo(); anti = Antifraude(); libro = Libro(); notif = Notificador()

origen = {"id": "A1", "saldo": 500000}
destino = {"id": "B2", "saldo": 0}
monto = 200000

if not val.hay_fondos(origen, monto):
    print("Sin fondos")
elif not anti.es_seguro(origen, destino, monto):
    print("Bloqueado por antifraude")
else:
    origen["saldo"] -= monto
    destino["saldo"] += monto
    libro.registrar(origen, destino, monto)
    notif.enviar(origen, f"Transferiste ${monto}")
    print("Transferencia OK, saldo origen:", origen["saldo"])
print(">> Problema: cada canal (app, cajero, web) debe repetir TODA esta orquestacion.")

   [contabilidad] registrado: A1->B2 $200000
   [notif] a A1: Transferiste $200000
Transferencia OK, saldo origen: 300000
>> Problema: cada canal (app, cajero, web) debe repetir TODA esta orquestacion.


### Análisis del problema
- El cliente necesita conocer 4 subsistemas y su **orden** de invocación.
- Esa lógica se **duplica** en cada canal (app, cajero, web).
- Si mañana se agrega un paso (p. ej. retención de impuestos), hay que corregirlo en
  todos lados. Alto acoplamiento y riesgo de olvidar un paso.

## Código *con patrón* (problema resuelto)
`FachadaTransferencias` orquesta internamente todos los subsistemas y expone un solo
método `transferir(...)`. Los clientes solo llaman a la fachada.

In [2]:
class ValidadorSaldo:
    def hay_fondos(self, origen, monto): return origen["saldo"] >= monto

class Antifraude:
    def es_seguro(self, origen, destino, monto): return monto <= 10_000_000

class Libro:
    def registrar(self, origen, destino, monto):
        print(f"   [contabilidad] registrado: {origen['id']}->{destino['id']} ${monto}")

class Notificador:
    def enviar(self, cuenta, texto):
        print(f"   [notif] a {cuenta['id']}: {texto}")


# FACADE: encapsula la orquestacion completa
class FachadaTransferencias:
    def __init__(self):
        self._val = ValidadorSaldo()
        self._anti = Antifraude()
        self._libro = Libro()
        self._notif = Notificador()

    def transferir(self, origen: dict, destino: dict, monto: float) -> str:
        if not self._val.hay_fondos(origen, monto):
            return "RECHAZADA: sin fondos"
        if not self._anti.es_seguro(origen, destino, monto):
            return "RECHAZADA: antifraude"
        origen["saldo"] -= monto
        destino["saldo"] += monto
        self._libro.registrar(origen, destino, monto)
        self._notif.enviar(origen, f"Transferiste ${monto}")
        return f"OK: nuevo saldo origen ${origen['saldo']}"


banco = FachadaTransferencias()
origen = {"id": "A1", "saldo": 500000}
destino = {"id": "B2", "saldo": 0}

# El cliente solo hace UNA llamada, sin conocer los subsistemas
print(banco.transferir(origen, destino, 200000))
print(banco.transferir(origen, destino, 999999999))  # antifraude
print(">> Solucion: cualquier canal llama transferir(...) sin repetir la orquestacion.")

   [contabilidad] registrado: A1->B2 $200000
   [notif] a A1: Transferiste $200000
OK: nuevo saldo origen $300000
RECHAZADA: sin fondos
>> Solucion: cualquier canal llama transferir(...) sin repetir la orquestacion.


### Verificación
- El cliente hace **una sola llamada** `banco.transferir(...)`.
- Toda la coordinación (saldo, antifraude, contabilidad, notificación) queda **dentro**
  de la fachada.
- Agregar un paso nuevo se hace en **un solo lugar**.

## UML del patrón Facade
```plantuml
@startuml
class FachadaTransferencias {
    - _val : ValidadorSaldo
    - _anti : Antifraude
    - _libro : Libro
    - _notif : Notificador
    + transferir(origen, destino, monto)
}
class ValidadorSaldo { + hay_fondos(origen, monto) }
class Antifraude { + es_seguro(origen, destino, monto) }
class Libro { + registrar(origen, destino, monto) }
class Notificador { + enviar(cuenta, texto) }
FachadaTransferencias --> ValidadorSaldo
FachadaTransferencias --> Antifraude
FachadaTransferencias --> Libro
FachadaTransferencias --> Notificador
@enduml
```

## ¿Por qué Facade y no otro patrón?
- El problema es la **complejidad de coordinar varios subsistemas**. Facade da un punto
  de entrada único y simple, que es justo lo que necesitamos.
- No es Adapter: no estamos traduciendo una interfaz incompatible, los subsistemas ya son
  nuestros; los estamos **simplificando y coordinando**.
- No es Mediator (comportamiento): no gestionamos comunicación bidireccional entre objetos
  que colaboran, solo damos una cara simple hacia afuera.
- Facade reduce el acoplamiento de los clientes con el subsistema y evita duplicar la
  orquestación en cada canal.